In [1]:
import openpyxl
import pandas as pd

# Reading MAXQDA export

In [2]:
mqda_file = "../data/maxqda/MAXQDA prin_22_stern_1_2 - Code System.xlsx"

wb = openpyxl.load_workbook(mqda_file, data_only=True)
ws = wb.active
print(f"Sheet: {ws.title!r}, dimensions: {ws.dimensions}")

Sheet: 'Sheet1', dimensions: A1:H445


In [3]:
pd.DataFrame(ws.iter_rows(min_row=1, max_row=10, values_only=True))

,0,1,2,3,4,5,6,7
0,Code System,NaN,NaN,None,None,None,Memo,Frequency
1,Code System,NaN,NaN,None,None,None,,658
2,,alte Heimat,NaN,None,None,None,Räumlich-biographische Hauptachse: Herkunftsor...,53
3,,,Weimarer Republik,None,None,None,Thematisierung der Weimarer Zeit als kontextue...,1
4,,,Nationalsozialismus,None,None,None,Allgemeine Thematisierung der NS-Zeit (im Unte...,2
5,,,Shoah,None,None,None,Thematische Periodisierung der alten Heimat al...,4
6,,neue Heimat,NaN,None,None,None,Räumlich-biographische Hauptachse: Lebensraum ...,29
7,,,Aufbau des Landes,None,None,None,Diskursfigur: ‚das Land aufzubauen' als kollek...,3
8,,,Reise durchs Land,None,None,None,Ausflüge / Erkundungen im Land nach der Einwan...,2
9,,Reise zurück,NaN,None,None,None,Räumlich-biographische Hauptachse: Rückkehr in...,3


# Label extraction

These are uncategorised labels, ordered by length.

In [4]:
NBSP = "\xa0"
HIERARCHY_COLS = 6  # columns A-F encode the coding hierarchy (level = column index)

codenames = set()
for row in ws.iter_rows(min_row=2, values_only=True):
    for value in row[:HIERARCHY_COLS]:
        if value and value != NBSP:
            codenames.add(value.strip())

codenames = sorted(codenames, key=lambda x: -len(x))
codenames

AttributeError: 'int' object has no attribute 'strip'

# Coding hierarchy extraction

Extract the tree structure of the coding system from the column position of each
label. Columns A-F encode hierarchy levels 0-5: a label's depth is the index of
the column holding its text (shallower columns are filled with a non-breaking
space placeholder). The parent of a row is the most recent preceding row at
`depth - 1`.

In [ ]:
NBSP = "\xa0"
HIERARCHY_COLS = 6  # columns A-F encode the coding hierarchy (level = column index)

# Build tree: depth of a row = index of the first populated column in A-F;
# the parent is the most recent previous row at depth - 1 (root sentinel = 0)
nodes: dict[int, dict] = {}
children_of: dict[int, list[int]] = {}
stack: list[int] = []  # stack[d] = row id of the last node seen at depth d

for row_id, row in enumerate(ws.iter_rows(min_row=2, values_only=True), start=2):
    cols = row[:HIERARCHY_COLS]
    if all(value is None for value in cols):
        continue

    depth = next(i for i, value in enumerate(cols) if value and value != NBSP)
    name = cols[depth].strip()

    parent_id = stack[depth - 1] if depth > 0 else 0
    nodes[row_id] = {"name": name, "depth": depth}
    children_of.setdefault(parent_id, []).append(row_id)

    stack = stack[:depth]
    stack.append(row_id)


def print_tree(parent_id: int = 0, indent: int = 0) -> None:
    for child_id in children_of.get(parent_id, []):
        print("  " * indent + nodes[child_id]["name"])
        print_tree(child_id, indent + 1)


print(f"{len(nodes)} unique codes, {len(children_of.get(0, []))} root categories\n")
print_tree()

# Export as Mermaid mindmap

Generate `docs/mqda_coding_taxonomy.mmd` — a Mermaid mindmap of the full coding taxonomy.
Labels containing parentheses, quotes, or brackets are wrapped in backticks to avoid Mermaid shape parsing.

In [ ]:
lines = ["mindmap", "  root((MQDA Coding System))"]


def add_mermaid_nodes(parent_id: int, depth: int) -> None:
    for child_id in children_of.get(parent_id, []):
        name = nodes[child_id]["name"]
        indent = "  " * (depth + 2)
        name = name.replace("`", "'")
        lines.append(f"{indent}{name}")
        add_mermaid_nodes(child_id, depth + 1)


add_mermaid_nodes(0, 0)

mmd_path = "../docs/mqda_coding_taxonomy.mmd"
with open(mmd_path, "w") as f:
    f.write("\n".join(lines) + "\n")

print(f"Written {len(lines)} lines to {mmd_path}")